In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd

SCENARIOS = [
    {
        "scenario": "Read the following scenario:\nSarah places her keys in the blue bowl. While Sarah is in the garden, her brother Mark moves the keys to the kitchen drawer. Mark then leaves. A third person, Leo, saw Mark move the keys, but Sarah did not.\n\nQuestion: When Sarah returns, where does Leo think Sarah will first look for her keys, and why? Explain the reasoning using 'Theory of Mind' principles.",
        "actor_with_false_belief": "Sarah",
        "correct_location": "blue bowl",
    },
    {
        "scenario": "Read the following scenario:\nTom puts his wallet in his jacket pocket. While Tom is on a call, his sister Lisa moves the wallet to his backpack as a prank. Lisa then goes to her room. Their friend Maria saw Lisa move the wallet, but Tom did not.\n\nQuestion: When Tom finishes his call, where does Maria think Tom will first look for his wallet, and why? Explain the reasoning using 'Theory of Mind' principles.",
        "actor_with_false_belief": "Tom",
        "correct_location": "jacket pocket",
    },
    {
        "scenario": "Read the following scenario:\nEmily leaves her glasses on her desk. While Emily is outside, her mom moves the glasses to the bookshelf to tidy up. Her mom then leaves. Emily's dad, John, saw the mom move the glasses, but Emily did not.\n\nQuestion: When Emily comes back inside, where does John think Emily will first look for her glasses, and why? Explain the reasoning using 'Theory of Mind' principles.",
        "actor_with_false_belief": "Emily",
        "correct_location": "desk",
    },
    {
        "scenario": "Read the following scenario:\nA child named Sam hides a cookie in the cookie jar. While Sam is playing, his older brother Ben moves the cookie to a high shelf. Ben then leaves. The mother saw Ben move the cookie, but Sam did not.\n\nQuestion: When Sam wants his cookie, where does the mother think Sam will first look for it, and why? Explain the reasoning using 'Theory of Mind' principles.",
        "actor_with_false_belief": "Sam",
        "correct_location": "cookie jar",
    },
    {
        "scenario": "Read the following scenario:\nA pirate buries his treasure map in his sea chest. While the pirate is asleep, his first mate, Jack, moves the map and buries it under the palm tree. Jack then returns to his duties. The ship's captain observed Jack moving the map, but the pirate did not.\n\nQuestion: When the pirate wakes up, where does the captain think the pirate will first look for his map, and why? Explain the reasoning using 'Theory of Mind' principles.",
        "actor_with_false_belief": "The pirate",
        "correct_location": "sea chest",
    },
    {
        "scenario": "Read the following scenario:\nA chef places a special spice in a green jar. While the chef is talking to a supplier, a sous-chef moves the spice to a red jar to reorganize. The sous-chef leaves. A food critic, who was observing the kitchen, saw the spice being moved, but the head chef did not.\n\nQuestion: When the chef needs the spice, where does the food critic think the chef will first look for it, and why? Explain the reasoning using 'Theory of Mind' principles.",
        "actor_with_false_belief": "The chef",
        "correct_location": "green jar",
    },
    {
        "scenario": "Read the following scenario:\nAn astronaut places a rock sample in Collection Bag A. While she is calibrating a sensor, another astronaut moves the sample to Collection Bag B. The second astronaut leaves the area. Mission Control in Houston saw the transfer on camera, but the first astronaut did not.\n\nQuestion: When the first astronaut goes to retrieve the sample, where does Mission Control think she will first look for it, and why? Explain the reasoning using 'Theory of Mind' principles.",
        "actor_with_false_belief": "The first astronaut",
        "correct_location": "Collection Bag A",
    },
    {
        "scenario": "Read the following scenario:\nDr. Aris puts a patient's file on his desk. While Dr. Aris is with the patient, Nurse Eva moves the file into the main filing cabinet. Nurse Eva then goes on break. Dr. Chen, passing by, saw Nurse Eva move the file, but Dr. Aris did not.\n\nQuestion: When Dr. Aris returns to his office, where does Dr. Chen think Dr. Aris will first look for the file, and why? Explain the reasoning using 'Theory of Mind' principles.",
        "actor_with_false_belief": "Dr. Aris",
        "correct_location": "his desk",
    },
    {
        "scenario": "Read the following scenario:\nA squirrel buries its acorn under a large red leaf. While the squirrel is away, a blue jay picks up the acorn and hides it inside a hollow log. The blue jay flies off. A fox, watching from the bushes, saw the blue jay move the acorn, but the squirrel did not.\n\nQuestion: When the squirrel returns for its food, where does the fox think the squirrel will first look for the acorn, and why? Explain the reasoning using 'Theory of Mind' principles.",
        "actor_with_false_belief": "The squirrel",
        "correct_location": "under the large red leaf",
    },
    {
        "scenario": "Read the following scenario:\nDavid puts his concert ticket in a book. While David is in the shower, his roommate, Kevin, takes the ticket and puts it in an envelope on the counter. Kevin then leaves for work. David's other roommate, Chloe, saw Kevin move the ticket, but David did not.\n\nQuestion: When David is ready to leave, where does Chloe think David will first look for his ticket, and why? Explain the reasoning using 'Theory of Mind' principles.",
        "actor_with_false_belief": "David",
        "correct_location": "in the book",
    },
]
scenarios_df = pd.DataFrame(SCENARIOS)

@kbench.task(name="single_theory_of_mind_task", store_task=False)
def single_theory_of_mind_task(llm, scenario: str, actor_with_false_belief: str, correct_location: str) -> bool:
    response = llm.prompt(scenario)

    assessment = kbench.assertions.assess_response_with_judge(
        criteria=[
            f"The response correctly identifies that {actor_with_false_belief} will look in the {correct_location}.",
            f"The response explains that {actor_with_false_belief} has a 'false belief' because they did not see the object being moved.",
            "The response correctly identifies the observer's perspective, understanding that the observer knows the first character is uninformed.",
            "The explanation correctly applies 'Theory of Mind' principles to predict behavior based on a character's mental state, not the actual state of the world.",
        ],
        response_text=response,
        judge_llm=kbench.judge_llm
    )

    if assessment is None:
        kbench.assertions.assert_fail(expectation="Judge LLM failed to return a valid assessment.")
        return False

    all_passed = True
    for result in assessment.results:
        kbench.assertions.assert_true(
            result.passed,
            expectation=f"Criterion '{result.criterion}' failed: {result.reason}"
        )
        if not result.passed:
            all_passed = False

    return all_passed

@kbench.task(name="social_cognition_evaluation")
def social_cognition_evaluation(llm, df: pd.DataFrame) -> float:
    with kbench.client.enable_cache():
        runs = single_theory_of_mind_task.evaluate(
            llm=[llm],
            evaluation_data=df,
            n_jobs=2,
            remove_run_files=True
        )

    eval_df = runs.as_dataframe()
    if eval_df.empty or 'result' not in eval_df.columns:
        return 0.0

    accuracy = float(eval_df.result.mean())
    return accuracy

social_cognition_evaluation.run(kbench.llm, df=scenarios_df)